# EEG Eye-State · Causal Conv1D — Predict T+1 from Window T

Given **T consecutive EEG samples** across all 14 channels, predict whether  
the eyes will be **open or closed at the very next sample** (T+1).

```
t-T+1   t-T+2  ...   t          t+1
  |       |     ...   |            |
 [————————— input window ————————] → predict label here
   (14 channels × T timesteps)
```

This is a **causal, one-step-ahead** classification:  
the model never sees the future, only the T samples immediately before the target.

### Key differences from previous notebooks

| | Sample-level MLP | This notebook |
|---|---|---|
| Input | 14 values at time *t* | **14 × T values** ending at time *t* |
| Architecture | MLP (no temporal structure) | **Conv1D** (learns temporal filters) |
| Prediction target | label at *t* | **label at *t+1*** |
| Optuna searches | depth, width, dropout, lr | + **T (window size)**, kernel sizes, n_filters |


## 1 · Imports & Setup

In [1]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from mne.filter import filter_data
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib

matplotlib.use("Agg")
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

# colour palette (reused across all plots)
DARK = "#0a0e17"
CARD = "#111827"
EDGE = "#1f2937"
TEAL = "#00e5cc"
CORAL = "#ff4f5e"
GOLD = "#ffc947"
LIME = "#a8ff3e"
WHITE = "#f0f4ff"
GRAY = "#6b7280"

In [2]:
# ❌ before — only checks for CUDA
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ after — checks CUDA first, then MPS, then CPU
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")

Device: mps


## 2 · Load Dataset

In [3]:
print("Loading dataset …")
FS = 250

subject = 0
tache = "spt"
trial = 5

#  data_path = "/Users/vinicius/.cache/kagglehub/datasets/robikscube/eye-state-classification-eeg-dataset/versions/1"
data_path = f"../data/subject{subject}/"

# df = load_arff(os.path.join(data_path, "EEG_DATA.arff"))
df = pd.read_csv(
    f"{data_path}/subject_{subject}_tache_{tache}_trial_{trial}.csv", sep=","
)

EEG_COLS = [c for c in df.columns if c != "button" and c != "timestamp"]
TARGET = "button"
N_CHANNELS = len(EEG_COLS)
print(f"  Shape : {df.shape}")
print(f"  Channels ({len(EEG_COLS)}): {EEG_COLS}")
print(f"  Class balance:\n{df[TARGET].value_counts()}\n")

Loading dataset …
  Shape : (17197, 18)
  Channels (16): ['ch1', 'ch2', 'ch3', 'ch4', 'ch5', 'ch6', 'ch7', 'ch8', 'ch9', 'ch10', 'ch11', 'ch12', 'ch13', 'ch14', 'ch15', 'ch16']
  Class balance:
button
0    13806
1     3391
Name: count, dtype: int64



## 3 · Artifact Removal

Interpolate samples outside the `[q_lower, q_upper]` quantile range  
per channel — catches electrode pops and movement spikes.  
Set `APPLY_CLEANING = False` to skip.


In [4]:
APPLY_CLEANING = False
Q_UPPER, Q_LOWER, MARGIN = 0.999, 0.001, 5


def remove_artifacts_interpolate(df, eeg_cols, q_upper=0.999, q_lower=0.001, margin=5):
    """
    Flag samples beyond [q_lower, q_upper] quantiles per channel,
    expand the bad mask by `margin` samples on each side,
    then linearly interpolate across the gap.
    """
    df_c = df.copy()
    total = 0
    for ch in eeg_cols:
        col = df_c[ch].copy().astype(float)
        upper = col.quantile(q_upper)
        lower = col.quantile(q_lower)
        bad = (col > upper) | (col < lower)
        bad_exp = bad.copy()
        for s in range(1, margin + 1):
            bad_exp |= bad.shift(s, fill_value=False)
            bad_exp |= bad.shift(-s, fill_value=False)
        col[bad_exp] = np.nan
        df_c[ch] = col.interpolate(method="linear", limit_direction="both")
        total += bad_exp.sum()
    print(f"Replaced {total} sample-channel values.")
    return df_c


if APPLY_CLEANING:
    df = remove_artifacts_interpolate(df, EEG_COLS, Q_UPPER, Q_LOWER, MARGIN)
else:
    print("Artifact removal skipped.")


df.iloc[:, 1:-1] = filter_data(df.iloc[:, 1:-1].values.T, FS, 0.1, 60).T

Artifact removal skipped.
Setting up band-pass filter from 0.1 - 60 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 60.00 Hz
- Upper transition bandwidth: 15.00 Hz (-6 dB cutoff frequency: 67.50 Hz)
- Filter length: 8251 samples (33.004 s)



## 4 · Raw Data Extraction

Per-window normalisation inside `build_windows` handles all amplitude scaling — no global `StandardScaler` needed.

In [5]:
X_raw = df[EEG_COLS].values.astype(np.float32)
y_raw = df[TARGET].values.astype(int)
# per_window_norm inside build_windows handles all normalisation — no StandardScaler needed
N_CHANNELS = X_raw.shape[1]
print(f"X_raw : {X_raw.shape}  channels={N_CHANNELS}")

X_raw : (17197, 16)  channels=16


## 5 · Causal Sliding-Window Dataset Builder

For a given window length **T**, we build:

```
X_win[i] = X_raw[i : i+T]   →  per-window z-score  →  shape (C, T)
y_win[i] = y_raw[i+T]        →  label at the NEXT step
```

**Per-window normalisation** (`per_window_norm=True`): each window is
z-scored per channel independently before being passed to the model.
This removes session-level amplitude drift and makes the pipeline
identical to live inference — no global scaler needed.

`T` is part of the Optuna search space.

In [6]:
def build_windows(X: np.ndarray, y: np.ndarray, T: int,
                  per_window_norm: bool = True):
    """
    Construct causal windows of length T predicting one step ahead.

    Parameters
    ----------
    X : (N, C) raw EEG array
    y : (N,)   integer labels
    T : window length in samples
    per_window_norm : z-score each window per channel independently.
        Removes session-level amplitude drift — no global scaler needed.

    Returns
    -------
    X_win : (N-T, C, T)  — Conv1D expects (batch, channels, time)
    y_win : (N-T,)
    """
    N, C = X.shape
    n_windows = N - T
    X_win = np.empty((n_windows, C, T), dtype=np.float32)
    y_win = np.empty(n_windows, dtype=np.int64)

    for i in range(n_windows):
        w = X[i : i + T].copy()              # (T, C)
        if per_window_norm:
            mu = w.mean(axis=0, keepdims=True)
            sd = w.std(axis=0, keepdims=True) + 1e-8
            w  = (w - mu) / sd
        X_win[i] = w.T                        # (C, T) for Conv1D
        y_win[i] = y[i + T]                  # label at T+1

    return X_win, y_win


# Quick preview with T=64 (0.256 s at 250 Hz)
X_demo, y_demo = build_windows(X_raw, y_raw, T=64)
print(f"T=64  →  X_win {X_demo.shape}  y_win {y_demo.shape}")
print(f"Class balance: {np.bincount(y_demo)}  (closed / open)")

T=64  →  X_win (17133, 16, 64)  y_win (17133,)
Class balance: [13742  3391]  (closed / open)


## 6 · Causal Conv1D Architecture

```
Input  (batch, 14, T)
  │
  ├─ Conv1d(n_filters, kernel_size, padding='causal') + BN + act + Dropout
  ├─ Conv1d(...)  × (n_layers - 1)
  │
  ├─ AdaptiveAvgPool1d(1)   ← collapse time → (batch, last_filters, 1)
  │
  └─ Linear(last_filters, 1)  → logit → σ → P(eyes open at T+1)
```

**Causal padding** ensures no future leakage:  
each output timestep only attends to current and past inputs.

The number of filters can grow with depth (`grow_filters=True`)  
or stay constant — both are searchable.


In [7]:
class CausalConv1D(nn.Module):
    """
    Stack of causal Conv1D layers followed by global average pooling
    and a linear classifier head.

    Parameters
    ----------
    in_channels  : number of EEG channels (14)
    n_filters    : filters in the first conv layer
    n_layers     : number of conv layers
    kernel_size  : temporal kernel size (shared across layers)
    grow_filters : if True, double filters every layer (up to 256)
    dropout_rate : dropout after each conv block
    activation   : 'relu' | 'leaky_relu' | 'elu' | 'gelu'
    use_bn       : batch normalisation after each conv
    """

    ACTIVATIONS = {
        "relu": nn.ReLU,
        "leaky_relu": nn.LeakyReLU,
        "elu": nn.ELU,
        "gelu": nn.GELU,
    }

    def __init__(
        self,
        in_channels,
        n_filters,
        n_layers,
        kernel_size,
        grow_filters,
        dropout_rate,
        activation,
        use_bn,
    ):
        super().__init__()
        Act = self.ACTIVATIONS[activation]
        layers = []
        in_ch = in_channels

        for i in range(n_layers):
            out_ch = min(n_filters * (2**i) if grow_filters else n_filters, 256)
            padding = kernel_size - 1  # causal: pad left only

            layers.append(
                nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=padding)
            )
            layers.append(_Trim(padding))  # remove the right-side padding
            if use_bn:
                layers.append(nn.BatchNorm1d(out_ch))
            layers.append(Act())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            in_ch = out_ch

        self.conv_stack = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(in_ch, 1)
        self.last_ch = in_ch

    def forward(self, x):
        # x: (batch, 14, T)
        x = self.conv_stack(x)  # (batch, last_ch, T)
        x = self.pool(x).squeeze(-1)  # (batch, last_ch)
        return self.head(x).squeeze(1)


class _Trim(nn.Module):
    """Remove `n` timesteps from the right to enforce causal padding."""

    def __init__(self, n):
        super().__init__()
        self.n = n

    def forward(self, x):
        return x[..., : -self.n] if self.n > 0 else x


# Sanity check
_demo = CausalConv1D(
    N_CHANNELS,
    n_filters=32,
    n_layers=3,
    kernel_size=7,
    grow_filters=True,
    dropout_rate=0.3,
    activation="elu",
    use_bn=True,
)
_x = torch.randn(8, N_CHANNELS, 64)

print(f"Input : {_x.shape}")
print(f"Output: {_demo(_x).shape}")
total = sum(p.numel() for p in _demo.parameters())
print(f"Params: {total:,}")

Input : torch.Size([8, 16, 64])
Output: torch.Size([8])
Params: 76,065


## 7 · Training & Evaluation Helpers

In [8]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE).float()
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()


@torch.no_grad()
def eval_auc(model, X_t, y_t):
    model.eval()
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=512)
    probs = torch.cat(
        # ✅ fix 3 — xb to device before model call
        [torch.sigmoid(model(xb.to(DEVICE))).cpu() for xb, _ in loader]
    ).numpy()
    return float(roc_auc_score(y_t.cpu().numpy(), probs))


def build_optimizer(model, name, lr, wd):
    if name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    if name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    return torch.optim.SGD(model.parameters(), lr=lr, weight_decay=wd, momentum=0.9)


def run_fold(
    X_tr,
    y_tr,
    X_va,
    y_va,
    n_filters,
    n_layers,
    kernel_size,
    grow_filters,
    dropout,
    activation,
    use_bn,
    opt_name,
    lr,
    wd,
    batch_size,
    max_epochs=40,
    patience=8,
):
    """Train one fold; return (best_auc, preds, probs, history)."""
    pw = (
        torch.tensor([(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)]).float().to(DEVICE)
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    model = CausalConv1D(
        X_tr.shape[1],
        n_filters,
        n_layers,
        kernel_size,
        grow_filters,
        dropout,
        activation,
        use_bn,
    ).to(DEVICE)
    optimizer = build_optimizer(model, opt_name, lr, wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    loader = DataLoader(
        TensorDataset(X_tr, y_tr.float()), batch_size=batch_size, shuffle=True
    )

    best_auc, best_state, pat_cnt, history = 0.0, None, 0, []

    for _ in range(max_epochs):
        train_epoch(model, loader, optimizer, criterion)
        scheduler.step()
        auc = eval_auc(model, X_va, y_va)
        history.append(auc)
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat_cnt = 0
        else:
            pat_cnt += 1
        if pat_cnt >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        probs = torch.cat(
            [
                torch.sigmoid(model(xb.to(DEVICE))).cpu()
                for xb, _ in DataLoader(TensorDataset(X_va, y_va.float()), 512)
            ]
        ).numpy()
    return best_auc, (probs >= 0.5).astype(int), probs, history


print("Helpers defined.")

Helpers defined.


## 8 · Optuna Hyperparameter Search

### Search space

| Hyperparameter | Range / Choices | Notes |
|---|---|---|
| **T** (window length) | 16 – 256 samples | 0.125 s – 2 s of context |
| `n_layers` | 1 – 5 | conv depth |
| `n_filters` | 8 – 128 (log) | filters in first layer |
| `kernel_size` | 3, 5, 7, 11, 15 | temporal filter width |
| `grow_filters` | True / False | double filters each layer |
| `dropout` | 0.0 – 0.5 | |
| `activation` | relu, leaky_relu, elu, gelu | |
| `use_bn` | True / False | |
| `optimizer` | adam, adamw, sgd | |
| `lr` | 1e-4 – 1e-2 (log) | |
| `weight_decay` | 1e-5 – 1e-2 (log) | |
| `batch_size` | 64, 128, 256, 512 | |

> **Note:** T is rebuilt inside each trial — Optuna treats it as just  
> another hyperparameter and will find the optimal look-back duration.


In [ ]:
N_TRIALS = 20
INNER_FOLDS = 3
from sklearn.model_selection import GroupKFold as _GKF
cv_inner = _GKF(n_splits=INNER_FOLDS)


def objective(trial):
    # ── window ────────────────────────────────────────────────────────
    T = trial.suggest_int("T", 16, 256, log=True)

    # ── architecture ──────────────────────────────────────────────────
    n_layers     = trial.suggest_int("n_layers", 1, 5)
    n_filters    = trial.suggest_int("n_filters", 8, 128, log=True)
    kernel_size  = trial.suggest_categorical("kernel_size", [3, 5, 7])
    grow_filters = trial.suggest_categorical("grow_filters", [True, False])
    dropout      = trial.suggest_float("dropout", 0.0, 0.5)
    activation   = trial.suggest_categorical("activation", ["relu"])
    use_bn       = trial.suggest_categorical("use_bn", [True, False])

    # ── optimiser ─────────────────────────────────────────────────────
    opt_name   = trial.suggest_categorical("optimizer", ["adam"])
    lr         = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    wd         = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128])

    X_win, y_win = build_windows(X_raw, y_raw, T)
    X_t = torch.tensor(X_win)
    y_t = torch.tensor(y_win)

    # 1-second blocks keep windows with T-1 overlapping samples in the
    # same fold, preventing them from leaking across train/val boundary.
    block_ids = np.arange(len(y_win)) // 250

    fold_aucs = []
    for fold, (tr_idx, va_idx) in enumerate(
        cv_inner.split(X_win, y_win, groups=block_ids)
    ):
        auc, _, _, _ = run_fold(
            X_t[tr_idx], y_t[tr_idx], X_t[va_idx], y_t[va_idx],
            n_filters, n_layers, kernel_size, grow_filters,
            dropout, activation, use_bn, opt_name, lr, wd, batch_size,
        )
        fold_aucs.append(auc)
        trial.report(np.mean(fold_aucs), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_aucs))


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=0),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest trial : #{study.best_trial.number}")
print(f"Best AUC   : {study.best_value:.4f}")
print("\nBest params:")
for k, v in study.best_params.items():
    print(f"  {k:20s}: {v}")

## 9 · Extract Best Configuration

In [ ]:
p = study.best_params

T            = p["T"]
n_layers     = p["n_layers"]
n_filters    = p["n_filters"]
kernel_size  = p["kernel_size"]
grow_filters = p["grow_filters"]
dropout      = p["dropout"]
activation   = p["activation"]
use_bn       = p["use_bn"]
opt_name     = p["optimizer"]
lr           = p["lr"]
wd           = p["weight_decay"]
batch_size   = p["batch_size"]

# Build final windowed dataset from raw data with best T
X_win, y_win = build_windows(X_raw, y_raw, T)
X_torch = torch.tensor(X_win).to(DEVICE)
y_torch = torch.tensor(y_win).to(DEVICE)

print(f"Best T       : {T} samples  ({T/FS*1000:.0f} ms of look-back)")
print(f"Dataset      : {X_win.shape}  →  {y_win.shape}")
print(f"Conv layers  : {n_layers}  ×  {n_filters} filters  kernel={kernel_size}")
print(f"Grow filters : {grow_filters}")
print(f"Dropout      : {dropout:.3f}   BN: {use_bn}")
print(f"Optimizer    : {opt_name}   lr={lr:.2e}   wd={wd:.2e}")
print(f"Batch size   : {batch_size}")

## 10 · Final Evaluation — 5-Fold CV

Because each prediction maps to a specific sample index (the T+1 target),  
we can reconstruct the **full prediction time-series** over all 14 980 samples  
(minus the first T which have no complete look-back window).


In [ ]:
from sklearn.metrics import roc_curve
from sklearn.model_selection import GroupKFold

block_ids = np.arange(len(y_win)) // 250
cv_outer  = GroupKFold(n_splits=5)
final_aucs = []
all_preds  = np.full(len(y_win), -1, dtype=int)
all_probs  = np.zeros(len(y_win), dtype=np.float32)
train_hists = []

for fold, (tr_idx, va_idx) in enumerate(cv_outer.split(X_win, y_win, groups=block_ids)):
    auc, preds, probs, hist = run_fold(
        X_torch[tr_idx], y_torch[tr_idx],
        X_torch[va_idx],  y_torch[va_idx],
        n_filters, n_layers, kernel_size, grow_filters,
        dropout, activation, use_bn, opt_name, lr, wd, batch_size,
        max_epochs=60, patience=10,
    )
    final_aucs.append(auc)
    all_preds[va_idx]  = preds
    all_probs[va_idx]  = probs
    train_hists.append(hist)
    print(f"Fold {fold+1}  AUC={auc:.4f}  epochs={len(hist)}")

mask = all_preds >= 0
print(f"\nMean AUC : {np.mean(final_aucs):.4f} ± {np.std(final_aucs):.4f}")
print(f"Accuracy : {(all_preds[mask] == y_win[mask]).mean():.4f}")
print()
print(classification_report(y_win[mask], all_preds[mask], target_names=["Closed", "Open"]))

# ── Optimal decision threshold (Youden's J on CV holdout) ─────────────────
fpr_cv, tpr_cv, thresholds_cv = roc_curve(y_win[mask], all_probs[mask])
best_thresh = float(thresholds_cv[np.argmax(tpr_cv - fpr_cv)])
preds_opt   = (all_probs[mask] >= best_thresh).astype(int)
print(f"Optimal threshold (Youden's J) : {best_thresh:.3f}")
print(f"Accuracy @ optimal threshold   : {(preds_opt == y_win[mask]).mean():.4f}")
print()
print(classification_report(y_win[mask], preds_opt, target_names=["Closed", "Open"]))

# ── Retrain on all data → final model for inference ───────────────────────
print("Retraining on full dataset …")
_, _, _, final_model_hist = run_fold(
    X_torch, y_torch, X_torch, y_torch,
    n_filters, n_layers, kernel_size, grow_filters,
    dropout, activation, use_bn, opt_name, lr, wd, batch_size,
    max_epochs=60, patience=60,        # no early stopping — run full 60 epochs
)
final_model = CausalConv1D(
    N_CHANNELS, n_filters, n_layers, kernel_size,
    grow_filters, dropout, activation, use_bn,
).to(DEVICE)
_pw = torch.tensor(
    [(y_torch == 0).sum().item() / max((y_torch == 1).sum().item(), 1)]
).float().to(DEVICE)
_criterion = nn.BCEWithLogitsLoss(pos_weight=_pw)
_optimizer = build_optimizer(final_model, opt_name, lr, wd)
_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(_optimizer, T_max=60)
_loader    = DataLoader(TensorDataset(X_torch, y_torch.float()),
                        batch_size=batch_size, shuffle=True)
for _ in range(60):
    train_epoch(final_model, _loader, _optimizer, _criterion)
    _scheduler.step()
final_model.eval()
print("Final model ready.")

## 11 · Plot A — All Channels + Prediction & Label Bars

Each EEG channel gets its own row.  
At the top: two thin colour bars spanning the full recording —  
- **True label** bar: black = eyes open, invisible = closed  
- **Prediction** bar: black = predicted open, invisible = predicted closed

This makes it easy to see where the model agrees or disagrees with the ground truth.


In [ ]:
EXCERPT_START = 9_500
EXCERPT_END   = 11_000

sig_start = EXCERPT_START + T
sig_end   = EXCERPT_END   + T

t_axis   = np.arange(sig_start, sig_end)
true_seg = y_win[EXCERPT_START:EXCERPT_END]
pred_seg = all_preds[EXCERPT_START:EXCERPT_END]
signal   = X_raw[sig_start:sig_end]          # raw signal for display
L = len(t_axis)

N_CH        = len(EEG_COLS)
BAR_H       = 0.4
CH_H        = 1.0
N_ROWS      = 2 + N_CH
row_heights = [BAR_H, BAR_H] + [CH_H] * N_CH

fig, axes = plt.subplots(
    N_ROWS, 1, figsize=(18, 2.0 + N_CH * 1.1), facecolor=DARK,
    gridspec_kw={"height_ratios": row_heights, "hspace": 0.08},
)
fig.suptitle(
    f"EEG Eye-State · Conv1D Causal Prediction  "
    f"(T={T} samples = {T/FS*1000:.0f} ms look-back)",
    color=WHITE, fontsize=14, fontweight="bold", y=1.005,
)

x = np.arange(L)


def draw_bar(ax, labels, colour, title):
    ax.set_facecolor(DARK); ax.set_xlim(0, L); ax.set_ylim(0, 1)
    ax.set_yticks([]); ax.set_xticks([])
    for sp in ax.spines.values(): sp.set_color(EDGE)
    in_block, block_start = False, 0
    for i, v in enumerate(labels):
        if v == 1 and not in_block:
            block_start = i; in_block = True
        elif v == 0 and in_block:
            ax.axvspan(block_start, i, color=colour, alpha=1.0, linewidth=0)
            in_block = False
    if in_block:
        ax.axvspan(block_start, L, color=colour, alpha=1.0, linewidth=0)
    ax.set_ylabel(title, color=WHITE, fontsize=8, rotation=0,
                  ha="right", va="center", labelpad=60)


draw_bar(axes[0], true_seg, colour=WHITE, title="True\nlabel")
draw_bar(axes[1], pred_seg, colour=CORAL, title="Predicted")
for i, err in enumerate(true_seg != pred_seg):
    if err:
        axes[1].axvspan(i, i + 1, color=GOLD, alpha=0.6, linewidth=0)

for ch_i, (ax, ch_name) in enumerate(zip(axes[2:], EEG_COLS)):
    sig_ch = signal[:, ch_i]
    ax.set_facecolor(DARK)
    ax.plot(x, sig_ch, color=TEAL, lw=0.6, alpha=0.85)
    for i in range(L - 1):
        if true_seg[i] == 1:
            ax.axvspan(i, i + 1, alpha=0.08, color=LIME, linewidth=0)
    ax.set_xlim(0, L); ax.set_yticks([]); ax.set_xticks([])
    for sp in ax.spines.values(): sp.set_color(EDGE)
    ax.set_ylabel(ch_name, color=WHITE, fontsize=8, rotation=0,
                  ha="right", va="center", labelpad=5)

tick_samples = np.linspace(0, L, 7, dtype=int)
tick_seconds = np.round((tick_samples + sig_start) / FS, 1)
axes[-1].set_xticks(tick_samples)
axes[-1].set_xticklabels([f"{s}s" for s in tick_seconds], color=WHITE, fontsize=8)

from matplotlib.patches import Patch
fig.legend(
    handles=[
        Patch(facecolor=WHITE, label="Eyes open (true)"),
        Patch(facecolor=CORAL, label="Eyes open (pred)"),
        Patch(facecolor=GOLD,  label="Mismatch"),
        Patch(facecolor=LIME,  alpha=0.4, label="Open region (bg)"),
    ],
    loc="lower center", ncol=4, facecolor=CARD,
    labelcolor=WHITE, edgecolor=EDGE, fontsize=9,
    bbox_to_anchor=(0.5, -0.03),
)
plt.tight_layout()
plt.savefig("plot_signals_with_bars.png", dpi=150, bbox_inches="tight", facecolor=DARK)
plt.show()
print("Saved → plot_signals_with_bars.png")

## 12 · Plot B — Optuna Optimisation History

In [ ]:
trials_df = study.trials_dataframe(attrs=("number", "value", "params", "state"))
comp = trials_df[trials_df["state"] == "COMPLETE"].sort_values("number")
vals = comp["value"].values
nums = comp["number"].values
best_sf = np.maximum.accumulate(vals)

fig, ax = plt.subplots(figsize=(12, 4), facecolor=DARK)
ax.set_facecolor(CARD)
for s in ax.spines.values():
    s.set_color(EDGE)

ax.scatter(nums, vals, color=TEAL, alpha=0.5, s=35, zorder=3, label="Trial AUC")
ax.plot(nums, best_sf, color=GOLD, lw=2.5, zorder=4, label="Best so far")
ax.axhline(
    best_sf[-1],
    color=CORAL,
    lw=1.2,
    ls="--",
    label=f"Best = {best_sf[-1]:.4f}",
    alpha=0.8,
)
ax.set_xlabel("Trial", color=WHITE, fontsize=11)
ax.set_ylabel("Val AUC (3-fold)", color=WHITE, fontsize=11)
ax.set_title(
    "Optuna Optimisation History — Conv1D Causal Classifier",
    color=WHITE,
    fontsize=13,
    fontweight="bold",
)
ax.tick_params(colors=WHITE)
ax.yaxis.grid(True, color=EDGE, lw=0.8)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=10)
plt.tight_layout()
plt.savefig("plot_optuna_history.png", dpi=150, bbox_inches="tight", facecolor=DARK)
plt.show()

## 13 · Plot C — Window Size T vs AUC

Across all completed Optuna trials, scatter T against the achieved AUC.  
This reveals how sensitive performance is to the look-back duration.


In [ ]:
col_T = "params_T"
fig, ax = plt.subplots(figsize=(10, 4), facecolor=DARK)
ax.set_facecolor(CARD)
for s in ax.spines.values():
    s.set_color(EDGE)

if col_T in comp.columns:
    T_vals = comp[col_T].dropna().values
    auc_vals = comp.loc[comp[col_T].notna(), "value"].values
    sc = ax.scatter(
        T_vals / FS * 1000,
        auc_vals,
        c=auc_vals,
        cmap="plasma",
        s=55,
        alpha=0.85,
        zorder=3,
    )
    ax.axvline(
        T / FS * 1000,
        color=CORAL,
        lw=2.5,
        ls="--",
        label=f"Best T = {T} samples ({T/FS*1000:.0f} ms)",
    )
    plt.colorbar(sc, ax=ax, label="AUC")

ax.set_xlabel("Window Length T  (ms)", color=WHITE, fontsize=11)
ax.set_ylabel("Validation AUC", color=WHITE, fontsize=11)
ax.set_title(
    "Look-back Duration T vs Classification AUC",
    color=WHITE,
    fontsize=13,
    fontweight="bold",
)
ax.tick_params(colors=WHITE)
ax.yaxis.grid(True, color=EDGE, lw=0.8)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=10)
plt.tight_layout()
plt.savefig("plot_T_vs_auc.png", dpi=150, bbox_inches="tight", facecolor=DARK)
plt.show()

## 14 · Plot D — Per-Fold Training Curves

In [ ]:
fold_colors = [TEAL, GOLD, LIME, CORAL, "#aa88ff"]
fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK)
ax.set_facecolor(CARD)
for s in ax.spines.values():
    s.set_color(EDGE)

for fi, hist in enumerate(train_hists):
    ax.plot(hist, color=fold_colors[fi], lw=2, alpha=0.9, label=f"Fold {fi+1}")

ax.set_xlabel("Epoch", color=WHITE, fontsize=11)
ax.set_ylabel("Validation AUC", color=WHITE, fontsize=11)
ax.set_title(
    "Training Curves — Best Conv1D Config (5-fold CV)",
    color=WHITE,
    fontsize=13,
    fontweight="bold",
)
ax.tick_params(colors=WHITE)
ax.yaxis.grid(True, color=EDGE, lw=0.8)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=10)
plt.tight_layout()
plt.savefig("plot_training_curves.png", dpi=150, bbox_inches="tight", facecolor=DARK)
plt.show()

## 15 · Plot E — Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4), facecolor=DARK)
ax.set_facecolor(CARD)
for s in ax.spines.values():
    s.set_color(EDGE)

cm = confusion_matrix(y_win[mask], all_preds[mask])
disp = ConfusionMatrixDisplay(cm, display_labels=["Closed", "Open"])
disp.plot(ax=ax, colorbar=False, cmap="YlOrRd")

ax.set_title(
    f"Confusion Matrix  (T={T}, N={mask.sum():,} samples)",
    color=WHITE,
    fontsize=11,
    fontweight="bold",
)
ax.tick_params(colors=WHITE)
ax.xaxis.label.set_color(WHITE)
ax.yaxis.label.set_color(WHITE)
for txt in ax.texts:
    txt.set_color("black")
    txt.set_fontsize(13)
plt.tight_layout()
plt.savefig("plot_confusion_matrix.png", dpi=150, bbox_inches="tight", facecolor=DARK)
plt.show()

## 16 · Results Summary

In [ ]:
acc = (all_preds[mask] == y_win[mask]).mean()

print("=" * 60)
print("  EEG Eye-State — Causal Conv1D Results")
print("=" * 60)
print(f"  Look-back window : T = {T} samples  ({T/FS*1000:.0f} ms)")
print(f"  Predicts         : eye state at sample T+1")
print(f"  Dataset size     : {X_win.shape[0]:,} windows")
print()
print(f"  Best trial AUC   : {study.best_value:.4f}")
print(f"  CV AUC (5-fold)  : {np.mean(final_aucs):.4f} ± {np.std(final_aucs):.4f}")
print(f"  Accuracy         : {acc:.4f}")
print()
print("  Best Architecture")
print(f"    Conv layers    : {n_layers}")
print(f"    Base filters   : {n_filters}  grow={grow_filters}")
print(f"    Kernel size    : {kernel_size}")
print(f"    Activation     : {activation}")
print(f"    Dropout        : {dropout:.3f}   BN: {use_bn}")
print(f"    Optimizer      : {opt_name}  lr={lr:.1e}  wd={wd:.1e}")
print(f"    Batch size     : {batch_size}")
print("=" * 60)

## 17 · Predict on a New Trial

Apply the trained model to a held-out trial.

**Pipeline for new data:**
1. Band-pass filter (same settings as training)
2. Build windows with `per_window_norm=True` — no scaler, each window is self-normalised
3. Predict with `final_model` using the optimal threshold (Youden's J)

---

In [ ]:
print("Loading new trial …")

subject_new = 0
tache_new   = "spt"
trial_new   = 0

df_new = pd.read_csv(
    f"../data/subject{subject_new}/"
    f"subject_{subject_new}_tache_{tache_new}_trial_{trial_new}.csv"
)
EEG_COLS_NEW = [c for c in df_new.columns if c not in ("button", "timestamp")]
assert EEG_COLS_NEW == EEG_COLS, "Channel mismatch between train and new trial!"

df_new.iloc[:, 1:-1] = filter_data(df_new.iloc[:, 1:-1].values.T, FS, 0.1, 60).T

X_new = df_new[EEG_COLS].values.astype(np.float32)
y_new = df_new[TARGET].values.astype(int)

print(f"  Shape : {df_new.shape}")
print(f"  Class balance: {df_new[TARGET].value_counts().to_dict()}")

In [ ]:
# Build windows — per_window_norm handles normalisation, no scaler needed
X_win_new, y_win_new = build_windows(X_new, y_new, T, per_window_norm=True)
X_torch_new = torch.tensor(X_win_new).to(DEVICE)

print(f"New trial windows : {X_win_new.shape}")

In [ ]:
final_model.eval()
with torch.no_grad():
    loader_new  = DataLoader(TensorDataset(X_torch_new,
                             torch.zeros(len(X_torch_new))), batch_size=512)
    probs_new   = torch.cat(
        [torch.sigmoid(final_model(xb.to(DEVICE))).cpu() for xb, _ in loader_new]
    ).numpy()

preds_new = (probs_new >= best_thresh).astype(int)
auc_new   = roc_auc_score(y_win_new, probs_new)
acc_new   = (preds_new == y_win_new).mean()

print(f"New trial {trial_new} — AUC : {auc_new:.4f}   Accuracy : {acc_new:.4f}"
      f"  (threshold={best_thresh:.3f})")
print()
print(classification_report(y_win_new, preds_new, target_names=["Closed", "Open"]))

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(3, 1, figsize=(18, 6), sharex=True, facecolor=DARK)
t_ax = np.arange(len(probs_new))

ax = axes[0]
ax.set_facecolor(CARD)
ax.plot(t_ax, probs_new, color=TEAL, lw=0.8, label="P(open)")
ax.axhline(best_thresh, color=CORAL, lw=1.2, ls="--",
           label=f"Threshold={best_thresh:.3f}")
ax.set_ylim(0, 1); ax.set_ylabel("P(open)", color=WHITE, fontsize=9)
ax.tick_params(colors=WHITE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)
for sp in ax.spines.values(): sp.set_color(EDGE)

ax = axes[1]
ax.set_facecolor(CARD)
ax.plot(t_ax, preds_new, drawstyle="steps-post", color=CORAL, lw=1.5, label="Predicted")
ax.plot(t_ax, y_win_new, drawstyle="steps-post", color=WHITE, lw=1.0,
        alpha=0.5, label="True")
ax.set_ylim(-0.1, 1.1); ax.set_yticks([0, 1])
ax.set_ylabel("Label", color=WHITE, fontsize=9)
ax.tick_params(colors=WHITE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)
for sp in ax.spines.values(): sp.set_color(EDGE)

ax = axes[2]
ax.set_facecolor(CARD)
errors = (preds_new != y_win_new).astype(float)
ax.fill_between(t_ax, errors, color=GOLD, alpha=0.7, step="post", label="Error")
ax.set_ylim(0, 1.2); ax.set_yticks([])
ax.set_ylabel("Error", color=WHITE, fontsize=9)
ax.set_xlabel("Sample", color=WHITE, fontsize=9)
ax.tick_params(colors=WHITE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)
for sp in ax.spines.values(): sp.set_color(EDGE)

fig.suptitle(
    f"New Trial (trial={trial_new}) · Conv1D  AUC={auc_new:.4f}  Acc={acc_new:.4f}",
    color=WHITE, fontsize=13, fontweight="bold"
)
plt.tight_layout()

In [ ]:
import json, os

SAVE_PATH = f"../data/subject{subject}/model_conv1d"
os.makedirs(SAVE_PATH, exist_ok=True)

torch.save(final_model.state_dict(), f"{SAVE_PATH}/eeg_conv1d_model.pt")

params_to_save = {
    "T":            int(T),
    "n_channels":   int(N_CHANNELS),
    "channel_names": EEG_COLS,
    "fs":           int(FS),
    "n_layers":     int(n_layers),
    "n_filters":    int(n_filters),
    "kernel_size":  int(kernel_size),
    "grow_filters": bool(grow_filters),
    "dropout":      float(dropout),
    "activation":   activation,
    "use_bn":       bool(use_bn),
    "best_thresh":  float(best_thresh),
    "cv_auc_mean":  float(np.mean(final_aucs)),
    "cv_auc_std":   float(np.std(final_aucs)),
    "per_window_norm": True,
}
with open(f"{SAVE_PATH}/eeg_conv1d_params.json", "w") as f:
    json.dump(params_to_save, f, indent=2)

print("Saved:")
print(f"  {SAVE_PATH}/eeg_conv1d_model.pt")
print(f"  {SAVE_PATH}/eeg_conv1d_params.json  (thresh={best_thresh:.3f})")

In [ ]:
%matplotlib inline
# ── Predict on all other trials ───────────────────────────────────────────
OTHER_TRIALS = [t for t in range(10) if t != trial]   # every trial except train

results = []
final_model.eval()

for t in OTHER_TRIALS:
    df_t = pd.read_csv(
        f"../data/subject{subject}/"
        f"subject_{subject}_tache_{tache}_trial_{t}.csv"
    )
    df_t.iloc[:, 1:-1] = filter_data(df_t.iloc[:, 1:-1].values.T, FS, 0.1, 60).T

    X_t = df_t[EEG_COLS].values.astype(np.float32)
    y_t = df_t[TARGET].values.astype(int)

    X_win_t, y_win_t = build_windows(X_t, y_t, T, per_window_norm=True)
    X_tor_t = torch.tensor(X_win_t).to(DEVICE)

    with torch.no_grad():
        loader_t  = DataLoader(TensorDataset(X_tor_t, torch.zeros(len(X_tor_t))),
                               batch_size=512)
        probs_t   = torch.cat(
            [torch.sigmoid(final_model(xb.to(DEVICE))).cpu() for xb, _ in loader_t]
        ).numpy()

    preds_t = (probs_t >= best_thresh).astype(int)
    auc_t   = roc_auc_score(y_win_t, probs_t)
    acc_t   = (preds_t == y_win_t).mean()
    results.append({"trial": t, "auc": auc_t, "acc": acc_t,
                    "probs": probs_t, "preds": preds_t, "y": y_win_t})
    print(f"  trial {t:2d}: AUC={auc_t:.4f}  Acc={acc_t:.4f}  n={len(y_win_t):,}")

aucs = [r["auc"] for r in results]
accs = [r["acc"] for r in results]
print(f"\nMean AUC : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"Mean Acc : {np.mean(accs):.4f} ± {np.std(accs):.4f}")

# ── Summary bar chart ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor=DARK)

for ax, values, ylabel, title in [
    (axes[0], aucs, "AUC",      "AUC per trial"),
    (axes[1], accs, "Accuracy", "Accuracy per trial"),
]:
    ax.set_facecolor(CARD)
    for sp in ax.spines.values(): sp.set_color(EDGE)
    bar_colours = [GOLD if v == max(values) else
                   CORAL if v == min(values) else TEAL for v in values]
    ax.bar(OTHER_TRIALS, values, color=bar_colours, edgecolor=EDGE, alpha=0.9)
    ax.axhline(np.mean(values), color=WHITE, lw=1.5, ls="--",
               label=f"Mean={np.mean(values):.4f}")
    ax.set_xticks(OTHER_TRIALS)
    ax.set_xticklabels([f"T{t}" for t in OTHER_TRIALS], color=WHITE, fontsize=9)
    ax.set_ylabel(ylabel, color=WHITE, fontsize=10)
    ax.set_xlabel("Trial", color=WHITE, fontsize=10)
    ax.set_title(title, color=WHITE, fontsize=12, fontweight="bold")
    ax.tick_params(colors=WHITE); ax.yaxis.grid(True, color=EDGE, lw=0.6)
    ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=9)

plt.suptitle(
    f"Conv1D trained on trial {trial} · evaluated on {OTHER_TRIALS}",
    color=WHITE, fontsize=13, fontweight="bold"
)
plt.tight_layout()

# ── Per-trial probability timeline (small multiples) ─────────────────────
n_cols = 3
n_rows = int(np.ceil(len(OTHER_TRIALS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(6 * n_cols, 3 * n_rows),
                         facecolor=DARK)
axes = axes.flatten()

for ax, r in zip(axes, results):
    ax.set_facecolor(CARD)
    for sp in ax.spines.values(): sp.set_color(EDGE)
    t_ax = np.arange(len(r["probs"]))
    ax.plot(t_ax, r["probs"], color=TEAL, lw=0.5, alpha=0.8)
    ax.plot(t_ax, r["y"],     color=WHITE, lw=0.8, alpha=0.3,
            drawstyle="steps-post", label="True")
    ax.axhline(best_thresh, color=CORAL, lw=1, ls="--")
    ax.set_ylim(0, 1); ax.set_yticks([0, 0.5, 1])
    ax.tick_params(colors=WHITE, labelsize=7)
    ax.set_title(f"Trial {r['trial']}  AUC={r['auc']:.3f}  Acc={r['acc']:.3f}",
                 color=WHITE, fontsize=9, fontweight="bold")

for ax in axes[len(results):]:   # hide unused subplots
    ax.set_visible(False)

plt.suptitle("P(open) per trial — Conv1D", color=WHITE, fontsize=13, fontweight="bold")
plt.tight_layout()